# AAX Audio Converter — Colab transcription

This notebook transcribes the audiobook audio exported by **AAX Audio Converter**
(Transcription tab → Engine: *Google Colab export*) using **faster-whisper** on a GPU,
and writes the Markdown transcript back next to the audio for import into NotebookLM.

**Runtime → Change runtime type → GPU** before running.

Workflow:
1. In the app, set *Colab export folder* to a folder that **Google Drive for Desktop** syncs.
2. Convert your book(s). The app writes `<folder>/<Author - Title>/audio/NNN.wav` + `manifest.json`.
3. Let Drive finish syncing, then run all cells here.
4. The transcript `<Author - Title>.md` (or one file per chapter) appears in each book folder and syncs back.


In [ ]:
!pip -q install faster-whisper
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Edit these, then Runtime -> Run all.
EXPORT_ROOT = '/content/drive/MyDrive/AaxColab'   # folder that holds the per-book export folders
MODEL_SIZE = 'large-v3'                            # base | small | medium | large-v3
COMPUTE_TYPE = 'float16'                           # float16 | int8_float16 | int8
FORCE_REPROCESS = False                            # re-run books that already have a .done marker


In [ ]:
import os, json, re, glob
from faster_whisper import WhisperModel

# Mirror of Transcriber.__boilerplate (AaxAudioConverterLib/Transcriber.cs). Keep in sync.
BOILERPLATE = [re.compile(p, re.IGNORECASE) for p in [
    'this is audible',
    'audible.com',
    'audible hopes',
    'audible original',
    'audible studios',
    'brought to you by audible',
]]
BOILERPLATE.append(re.compile('オーディブル'))

def filter_boilerplate(text):
    lines = [l.strip() for l in text.splitlines()]
    kept = [l for l in lines if l and not any(rx.search(l) for rx in BOILERPLATE)]
    return ' '.join(kept)

def heading(num, title):
    return f'{num}. {title}' if title else str(num)

print('loading model', MODEL_SIZE)
model = WhisperModel(MODEL_SIZE, device='cuda', compute_type=COMPUTE_TYPE)

def transcribe(path, lang):
    segments, info = model.transcribe(path, language=lang, vad_filter=True)
    return ' '.join(s.text.strip() for s in segments).strip()

def write_book_md(book_dir, manifest, order, results):
    name = os.path.basename(book_dir.rstrip('/'))
    path = os.path.join(book_dir, name + '.md')
    with open(path, 'w', encoding='utf-8') as f:
        print('# ' + (manifest.get('title') or name), file=f)
        print('', file=f)
        if manifest.get('author'):
            print('- **Author**: ' + manifest['author'], file=f)
        if manifest.get('narrator'):
            print('- **Narrator**: ' + manifest['narrator'], file=f)
        print('', file=f)
        for num in order:
            r = results[num]
            if not r['paras']:
                continue
            print('## ' + heading(num, r['title']), file=f)
            print('', file=f)
            for para in r['paras']:
                print(para, file=f)
                print('', file=f)
    print('wrote', path)

def write_chapter_md(book_dir, num, r):
    if not r['paras']:
        return
    path = os.path.join(book_dir, f'{num:03d}.md')
    with open(path, 'w', encoding='utf-8') as f:
        print('# ' + heading(num, r['title']), file=f)
        print('', file=f)
        for para in r['paras']:
            print(para, file=f)
            print('', file=f)
    print('wrote', path)

manifests = sorted(glob.glob(os.path.join(EXPORT_ROOT, '*', 'manifest.json')))
print('found', len(manifests), 'book(s)')
for mpath in manifests:
    book_dir = os.path.dirname(mpath)
    done = os.path.join(book_dir, '.done')
    if os.path.exists(done) and not FORCE_REPROCESS:
        print('skip (done):', book_dir)
        continue
    manifest = json.load(open(mpath, encoding='utf-8'))
    lang = manifest.get('language', 'en')
    do_filter = manifest.get('filterBoilerplate', True)
    mode = manifest.get('markdown', 'perBook')
    results = {}
    order = []
    for ch in manifest.get('chapters', []):
        audio = os.path.join(book_dir, ch.get('audio', ''))
        if not os.path.exists(audio):
            print('missing audio:', audio)
            continue
        text = transcribe(audio, lang)
        if do_filter:
            text = filter_boilerplate(text)
        num = ch.get('number')
        if num not in results:
            results[num] = {'title': ch.get('title'), 'paras': []}
            order.append(num)
        if text:
            results[num]['paras'].append(text)
    if mode == 'perChapter':
        for num in order:
            write_chapter_md(book_dir, num, results[num])
    else:
        write_book_md(book_dir, manifest, order, results)
    open(done, 'w').close()
print('all done')
